<a href="https://colab.research.google.com/github/jacielefreitas63-tech/projeto-delivery-logistic/blob/main/1_limpeza_e_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Projeto Delivery: Engenharia de Dados & Análise Exploratória (EDA)


# 1 Configuração do Ambiente e Conexão com o Data Lake (Google Drive)

Neste notebook, realizaremos a ingestão, limpeza fina, tratamento de outliers e união das bases transacionais da *Olist, malha rodoviária do **OpenStreetMap* e dados meteorológicos do *INMET*.

In [ ]:
!pip install osmnx networkx pandas numpy pyspark

In [ ]:
# Importação das bibliotecas fundamentais para manipulação e análise estatística
import pandas as pd
import numpy as np
import networkx as nx
import os as ox

# Imports específicos do PySpark para processamento distribuído
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
print("🚀 Bibliotecas carregadas com sucesso!")

🚀 Bibliotecas carregadas com sucesso!


In [ ]:
# Inicialização do motor do Spark
spark = SparkSession.builder \
    .appName("LogisticaDelivery") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("🚀 Spark inicializado com sucesso e pronto para processamento distribuído!")

🚀 Spark inicializado com sucesso e pronto para processamento distribuído!


In [ ]:
# Conexão segura com o armazenamento de dados do Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 📦2.Ingestão da Base de Pedidos (Olist Orders)
> O objetivo desta etapa é ler o dataset transacional de ordens de serviço, inspecionar os tipos de dados primitivos e mapear o volume inicial de registros antes de aplicar as regras de limpeza.

In [ ]:
# Caminho
base_path = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

In [ ]:
import pandas as pd
import os

# Mapeamento dos arquivos
arquivos = {
    'customers': 'olist_customers_dataset.csv',
    'items': 'olist_order_items_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv'
}

df_raw = {}

# Carregamento e Otimização em Tempo de Execução
for nome, arquivo in arquivos.items():
    caminho_completo = os.path.join(base_path, arquivo)

    if os.path.exists(caminho_completo):
        # Carrega o CSV
        df_raw[nome] = pd.read_csv(caminho_completo)


        print(f"✅ '{nome}' processado. Shape: {df_raw[nome].shape}")
    else:
        print(f"❌ Arquivo não encontrado: {arquivo}")

✅ 'customers' processado. Shape: (99441, 5)
✅ 'items' processado. Shape: (112650, 7)
✅ 'orders' processado. Shape: (99441, 8)
✅ 'geolocation' processado. Shape: (1000163, 5)
✅ 'reviews' processado. Shape: (99224, 7)


In [ ]:
 # Otimização: Converte objetos de baixa cardinalidade para 'category'
for col in df_raw[nome].select_dtypes(include=['object']).columns:
  if df_raw[nome][col].nunique() < 1000:
    df_raw[nome][col] = df_raw[nome][col].astype('category')

In [ ]:
  # Otimização: Downcast de numéricos para reduzir footprint de memória
cols_num = df_raw[nome].select_dtypes(include=['int64', 'float64']).columns
for col in cols_num:
    if df_raw[nome][col].dtype == 'int64':
        df_raw[nome][col] = pd.to_numeric(df_raw[nome][col], downcast='integer')
    else:
        df_raw[nome][col] = pd.to_numeric(df_raw[nome][col], downcast='float')

print(f"✅ '{nome}' processado. Shape: {df_raw[nome].shape}")

✅ 'reviews' processado. Shape: (99224, 7)


Tratamento Individual (Limpeza pré-merge)
Tratamos o CEP aqui para garantir que o merge funcione perfeitamente.

In [ ]:
# Tratamento de CEP (String com 5 dígitos)
df_raw['customers']['customer_zip_code_prefix'] = df_raw['customers']['customer_zip_code_prefix'].astype(str).str.zfill(5)
df_raw['geolocation']['geolocation_zip_code_prefix'] = df_raw['geolocation']['geolocation_zip_code_prefix'].astype(str).str.zfill(5)

# Criando versão única da geolocalização para evitar duplicatas no merge
df_geo_unica = df_raw['geolocation'].drop_duplicates(subset=['geolocation_zip_code_prefix'])

 Consolidação (Merge) das Bases

In [ ]:
# Base principal (Pedidos + Itens)
df_completo = pd.merge(df_raw['orders'], df_raw['items'], on='order_id', how='left')

# Adicionando Clientes
df_completo = pd.merge(df_completo, df_raw['customers'], on='customer_id', how='left')

# Adicionando Reviews
df_completo = pd.merge(df_completo, df_raw['reviews'][['order_id', 'review_score']], on='order_id', how='left')

# Adicionando Geolocalização
df_completo = pd.merge(df_completo, df_geo_unica, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')

Limpeza Final, Validação e Exportação
Finalizamos removendo nulos e validando os dados antes de salvar.

In [ ]:
# Tratamento de valores nulos
df_completo['review_score'] = df_completo['review_score'].fillna(-1)
df_completo = df_completo.dropna(subset=['geolocation_lat', 'geolocation_lng'])

# Conversão de datas
df_completo['order_purchase_timestamp'] = pd.to_datetime(df_completo['order_purchase_timestamp'])

# Validação Final
print("--- Validação de Nulos ---")
print(df_completo[['geolocation_lat', 'geolocation_lng']].isnull().sum())
df_completo.info()

# Exportação
df_completo.to_parquet(os.path.join(base_path, 'df_completo_final.parquet'))
print("✅ Pipeline concluído com sucesso!")

--- Validação de Nulos ---
geolocation_lat    0
geolocation_lng    0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 113781 entries, 0 to 114091
Data columns (total 24 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113781 non-null  object        
 1   customer_id                    113781 non-null  object        
 2   order_status                   113781 non-null  object        
 3   order_purchase_timestamp       113781 non-null  datetime64[ns]
 4   order_approved_at              113620 non-null  object        
 5   order_delivered_carrier_date   111812 non-null  object        
 6   order_delivered_customer_date  110546 non-null  object        
 7   order_estimated_delivery_date  113781 non-null  object        
 8   order_item_id                  113007 non-null  float64       
 9   product_id                     113007 non-null  object     

In [ ]:
print(df_completo.shape)
display(df_completo.head())

(113781, 24)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,review_score,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,...,7c396fd4830fd04220f754e42b4e5bff,03149,sao paulo,SP,4.0,03149,-23.574809,-46.587471,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,...,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,4.0,47813,-12.169860,-44.988369,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,...,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,5.0,75265,-16.746337,-48.514624,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,...,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,5.0,59296,-5.767733,-35.275467,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,...,72632f0f9dd73dfee390c9b22eb56dd6,09195,santo andre,SP,5.0,09195,-23.675037,-46.524784,santo andre,SP


## 🌤️3.Ingestão e Tratamento de Dados Meteorológicos (INMET)
> O objetivo desta etapa é carregar o histórico climático, tratar valores nulos de precipitação (chuva) e preparar a base para cruzamento temporal com os momentos de compra e entrega.

In [ ]:
# Caminho completo e direto
BASE_PATH = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

In [ ]:
# 1. Carregamento dos dados brutos do INMET
# Forçamos a leitura como STRING para otimizar o processo e evitar erros de inferência de esquema.
df_southeast = spark.read.option("header", "true").csv(BASE_PATH + 'southeast.csv')

#2. Verificando a estrutura
# Exibe o esquema e uma amostra para validar o carregamento inicial.
print("--- Estrutura do DataFrame Southeast ---")
df_southeast.printSchema()
df_southeast.show(5)

--- Estrutura do DataFrame Southeast ---
root
 |-- index: string (nullable = true)
 |-- Data: string (nullable = true)
 |-- Hora: string (nullable = true)
 |-- PRECIPITAÇÃO TOTAL, HORÁRIO (mm): string (nullable = true)
 |-- PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB): string (nullable = true)
 |-- PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB): string (nullable = true)
 |-- PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB): string (nullable = true)
 |-- RADIACAO GLOBAL (Kj/m²): string (nullable = true)
 |-- TEMPERATURA DO AR - BULBO SECO, HORARIA (°C): string (nullable = true)
 |-- TEMPERATURA DO PONTO DE ORVALHO (°C): string (nullable = true)
 |-- TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C): string (nullable = true)
 |-- TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C): string (nullable = true)
 |-- TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C): string (nullable = true)
 |-- TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C): string (nullable = true)
 |-- UMIDADE REL. MAX. NA HORA

Conversão de Tipos e Normalização
Esta etapa é crucial para garantir a integridade dos dados antes da limpeza. Convertendo as colunas numéricas e de data agora, evitamos erros de sintaxe e inconsistências durante os cálculos matemáticos e o cruzamento de informações (JOIN) com outras bases

In [ ]:
### 3. Limpeza de Placeholders (-9999)
# Substitui o valor de controle -9999 por NULL, garantindo a precisão estatística.

from pyspark.sql.functions import when, col

# Lista de colunas que precisam ser limpas
cols_para_limpar = [
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)"
]

# Aplicando a substituição para Southeast
for col_name in cols_para_limpar:
    df_southeast = df_southeast.withColumn(
        col_name,
        when(col(col_name) == -9999.0, None).otherwise(col(col_name))
    )

print("✅ Limpeza de placeholders (-9999) finalizada com sucesso.")

✅ Limpeza de placeholders (-9999) finalizada com sucesso.


📂 Tratamento e Salvamento da Base stations
A base foi carregada e salva no formato Parquet, garantindo uma estrutura otimizada para o futuro join com os dados climáticos.

In [ ]:
# 1. Ingestão Otimizada: Base Station
# Carregamos como String para evitar o custo computacional de inferência de esquema.
df_station = spark.read.option("header", "true").csv(BASE_PATH + 'stations.csv')

# Exibe o esquema inicial para confirmação dos nomes das colunas
print("--- Esquema da Base Station ---")
df_station.printSchema()

--- Esquema da Base Station ---
root
 |-- station: string (nullable = true)
 |-- region: string (nullable = true)
 |-- state: string (nullable = true)
 |-- station_code: string (nullable = true)
 |-- first date: string (nullable = true)
 |-- height: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- latitude: string (nullable = true)



Tratamento e Normalização stations
Nesta célula, aplicamos a limpeza necessária. Utilizei a função trim() para remover espaços invisíveis que costumam causar erros em joins (cruzamentos de tabelas).

In [ ]:
### 2. Tratamento da Base Station
# Limpeza de espaços em branco e garantia de tipos consistentes para o cruzamento de dados.

from pyspark.sql.functions import col, trim

# Remove espaços extras da coluna de identificação (ajuste o nome se for necessário)
# Mantendo o nome original da coluna conforme sua base.
df_station = df_station.withColumn("station_code", trim(col("station_code")))

# Exibe uma amostra dos dados tratados
print("✅ Base Station tratada com sucesso!")
df_station.show(5)

✅ Base Station tratada com sucesso!
+--------------------+------+-----+------------+----------+------+------------+------------+
|             station|region|state|station_code|first date|height|   longitude|    latitude|
+--------------------+------+-----+------------+----------+------+------------+------------+
|           QUEIMADAS|    NE|   BA|        A436|2008-05-23| 315.0|-39.61694443|  -10.984645|
|               MACAU|    NE|   RN|        A317|2007-01-06|  32.0|-36.57305554| -5.15111111|
|SAQUAREMA - SAMPA...|    SE|   RJ|        A667|2019-01-01|  26.0|-42.60888888| -22.8711111|
|SANTANA DO LIVRAM...|     S|   RS|        A804|2001-11-22| 328.0|-55.40138888|-30.75055555|
|          VILA VELHA|    SE|   ES|        A634|2017-02-15|  25.0|-40.40388888|-20.46694443|
+--------------------+------+-----+------------+----------+------+------------+------------+
only showing top 5 rows


In [ ]:
# 2. Salvar as estações tratadas
df_station.write.mode("overwrite").parquet(BASE_PATH + "stations_final.parquet")
print("✅ Estações tratadas e salvas!")

✅ Estações tratadas e salvas!


## 🗺️4.Ingestão e Tratamento de Dados de Malha Rodoviária (OpenStreetMap)
> Nesta etapa, mapeamos as coordenadas geográficas (latitude e longitude) dos clientes e vendedores para calcular as distâncias reais das rotas e entender o impacto do tráfego e das rodovias no tempo de entrega.

In [ ]:
### 1. Ingestão Otimizada: Malha Rodoviária
# Carregamos apenas as colunas essenciais para o cruzamento com bases de pedidos e clima.
# Isso economiza memória RAM drasticamente.

cols_necessarias = ["u", "v", "length", "highway", "speed_kph", "travel_time", "estado"] # Colunas presentes na base salva

df_malha_rodoviaria = spark.read.option("header", "true") \
    .parquet(BASE_PATH + 'malha_rodoviaria_particionada/') \
    .select(cols_necessarias)

print("✅ Base Malha Rodoviária carregada com seleção de colunas otimizada.")

✅ Base Malha Rodoviária carregada com seleção de colunas otimizada.


Como agora temos colunas de métricas logísticas (length, speed_kph, travel_time), vamos convertê-las para DoubleType para que você possa somar, calcular médias e identificar os gargalos de tempo de entrega.

In [ ]:
### 2. Limpeza e Conversão Unificada
# Aplicamos o trim nas colunas de texto (estado, highway) e o cast nas numéricas.

from pyspark.sql.functions import col, trim
from pyspark.sql.types import DoubleType

# Limpeza de campos de texto
df_malha_rodoviaria = df_malha_rodoviaria.withColumn("estado", trim(col("estado"))) \
                                         .withColumn("highway", trim(col("highway")))

# Conversão de métricas para DoubleType (essencial para cálculos de frete e tempo)
df_malha_rodoviaria = df_malha_rodoviaria.withColumn("length", col("length").cast(DoubleType())) \
                                         .withColumn("speed_kph", col("speed_kph").cast(DoubleType())) \
                                         .withColumn("travel_time", col("travel_time").cast(DoubleType()))

print("✅ Limpeza e conversão finalizadas.")
df_malha_rodoviaria.show(5)

✅ Limpeza e conversão finalizadas.
+--------+----------+------------------+------------+------------------+------------------+------+
|       u|         v|            length|     highway|         speed_kph|       travel_time|estado|
+--------+----------+------------------+------------+------------------+------------------+------+
|29040143|1974470329|142.71333162010592|     primary|  50.4697322312281|10.179724978102575|    SP|
|29040143|4779948178| 451.6755169933693|     primary|  50.4697322312281|32.217960930056684|    SP|
|29040143|7853095192|125.77849958427936|unclassified|41.182857142857145|10.994929199125295|    SP|
|29040144|8655992044| 41.58186864063914|     primary|  50.4697322312281|2.9660297467097996|    SP|
|29040144|4779897193| 57.26371299641137| residential|35.943018140179596|5.7354495380184245|    SP|
+--------+----------+------------------+------------+------------------+------------------+------+
only showing top 5 rows


Validação Final da Integridade
Garantimos que os dados estão "limpos" o suficiente para sustentar o cálculo de fretes e tempo de entrega.

In [ ]:
### 3. Validação de Qualidade
# Verificamos se existem valores nulos em colunas que usaremos para calcular o frete.

nulos_tempo = df_malha_rodoviaria.filter(col("travel_time").isNull()).count()

if nulos_tempo == 0:
    print("✅ Validação aprovada: Todos os registros possuem tempo de viagem válido.")
else:
    print(f"⚠️ Alerta: Identificamos {nulos_tempo} registros sem tempo de viagem (travel_time).")

✅ Validação aprovada: Todos os registros possuem tempo de viagem válido.


In [ ]:
# Lista as variáveis que o Spark reconhece
print([var for var in locals() if "df_" in var])

['df_raw', 'df_geo_unica', 'df_completo', 'df_southeast', 'df_station', 'df_malha_rodoviaria']


Salvamento com Particionamento
O uso do partitionBy organiza seu Data Lake em pastas, permitindo que o Spark leia apenas o que é necessário no futuro.

In [ ]:
### 3. Exportação: Salvando a base limpa em Parquet
# O comando abaixo salva o DataFrame de forma particionada por 'estado'.
# Isso organiza os dados em pastas, facilitando muito a leitura futura filtrada.

caminho_saida = BASE_PATH + 'malha_rodoviaria_limpa.parquet'

# Salvando com particionamento por estado para otimização de consultas
df_malha_rodoviaria.write \
    .mode("overwrite") \
    .partitionBy("estado") \
    .parquet(caminho_saida)

print(f"✅ Base salva com sucesso em formato Parquet no caminho: {caminho_saida}")

✅ Base salva com sucesso em formato Parquet no caminho: /content/drive/MyDrive/projeto_delivery_logistica/data/malha_rodoviaria_limpa.parquet


In [ ]:
# 1. Inspeção das Colunas (O seu "Dicionário de Dados")
# Rodar isso em cada base que você for cruzar.
print("--- Colunas disponíveis em df_orders ---")

# Define df_orders como um DataFrame PySpark para inspeção do esquema.
# Esta linha é adicionada para resolver o NameError caso df_orders não tenha sido definido pelas células anteriores.
df_orders = spark.createDataFrame(df_raw['orders'])

df_orders.printSchema()

# Se precisar ver o nome de todas as colunas como uma lista:
print(df_orders.columns)

--- Colunas disponíveis em df_orders ---
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


1. Integração de Dados Logísticos (Data Mart)

In [ ]:
### 1. Integração de Dados Logísticos (Data Mart)
# Consolidamos a base de pedidos com itens e clientes.
# Usamos 'left' join para garantir que não perderemos registros de pedidos.

df_order_items = spark.createDataFrame(df_raw['items'])
df_customers = spark.createDataFrame(df_raw['customers'])

df_final = df_orders \
    .join(df_order_items, on="order_id", how="left") \
    .join(df_customers, on="customer_id", how="left")

# Opcional: drop duplicatas se houver múltiplos itens para o mesmo order_id
df_final = df_final.dropDuplicates(["order_id"])

print("✅ Integração concluída com sucesso.")
print(f"Total de registros na base consolidada: {df_final.count()}")

✅ Integração concluída com sucesso.
Total de registros na base consolidada: 99441


## 🚚 5. Engenharia de Atributos (Feature Engineering): Métricas de SLA
> Criaremos as variáveis preditivas e de performance logística:
> 1. **tempo_entrega_dias**: Tempo real gasto desde a compra até a chegada ao cliente.
> 2. **atraso_entrega_dias**: Dias de atraso caso a entrega passe do prazo estimado (valores positivos indicam quebra de SLA).

In [ ]:
from pyspark.sql.functions import col, datediff, to_date, when, lit

# Converte df_completo (Pandas DataFrame) para PySpark DataFrame
df_analise = spark.createDataFrame(df_completo)

df_analise = df_analise \
    .withColumn("dt_entrega_real", to_date(when(col("order_delivered_customer_date") == 'NaN', lit(None)).otherwise(col("order_delivered_customer_date")))) \
    .withColumn("dt_entrega_estimada", to_date(when(col("order_estimated_delivery_date") == 'NaN', lit(None)).otherwise(col("order_estimated_delivery_date")))) \
    .withColumn("dias_atraso", datediff(col("dt_entrega_real"), col("dt_entrega_estimada")))

# Flag de Gargalo
df_analise = df_analise.withColumn("is_gargalo",
    when(col("dias_atraso") > 0, 1).otherwise(0))

print("✅ Métricas calculadas com sucesso no dataset limpo!")

✅ Métricas calculadas com sucesso no dataset limpo!


In [ ]:
### 2. Engenharia de Atributos: SLA e Gargalos
from pyspark.sql.functions import col, datediff, to_date, when, lit

# Criando as colunas de métricas de performance
df_final = df_final \
    .withColumn("dt_entrega_real", to_date(when(col("order_delivered_customer_date") == 'NaN', lit(None)).otherwise(col("order_delivered_customer_date")))) \
    .withColumn("dt_entrega_estimada", to_date(when(col("order_estimated_delivery_date") == 'NaN', lit(None)).otherwise(col("order_estimated_delivery_date")))) \
    .withColumn("dias_atraso", datediff(col("dt_entrega_real"), col("dt_entrega_estimada")))

# Criando a flag de gargalo (1 = atraso, 0 = no prazo)
df_final = df_final.withColumn("is_gargalo",
    when(col("dias_atraso") > 0, 1).otherwise(0))

print("✅ Métricas de SLA (dias_atraso, is_gargalo) criadas com sucesso.")
df_final.select("order_id", "dias_atraso", "is_gargalo").show(5)

✅ Métricas de SLA (dias_atraso, is_gargalo) criadas com sucesso.
+--------------------+-----------+----------+
|            order_id|dias_atraso|is_gargalo|
+--------------------+-----------+----------+
|00018f77f2f0320c5...|         -3|         0|
|000229ec398224ef6...|        -14|         0|
|00024acbcdf0a6daa...|         -6|         0|
|00042b26cf59d7ce6...|        -16|         0|
|00054e8431b9d7675...|        -17|         0|
+--------------------+-----------+----------+
only showing top 5 rows


Preparação das chaves de cruzamento
Para cruzar com o INMET e a Malha Rodoviária, precisamos garantir que o nosso df_final tenha uma coluna de referência comum (geralmente estado ou cep e data).

In [ ]:
# Preparação do INMET
from pyspark.sql.functions import col, lit, avg, max, to_date

precipitacao_col = "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)"

df_inmet_preparado = df_southeast \
    .withColumn(precipitacao_col, col(precipitacao_col).cast("double")) \
    .withColumn("data", to_date(when(col("Data") == 'NaN', lit(None)).otherwise(col("Data")))) \
    .withColumnRenamed("state", "estado") \
    .groupBy("estado", "data") \
    .agg(
        avg(col(precipitacao_col)).alias("media_chuva_mm"),
        max(col(precipitacao_col)).alias("max_chuva_mm")
    )

# Cache para acelerar
df_inmet_preparado.cache()
print("✅ INMET preparado.")

✅ INMET preparado.


In [ ]:
# Preparar Olist para o Join
df_final_preparado = df_final \
    .withColumnRenamed("customer_state", "estado") \
    .withColumn("data", to_date(col("order_purchase_timestamp")))

print("✅ Base Olist preparada.")

✅ Base Olist preparada.


 Reparticionamos para garantir que o Spark distribua os dados igualmente
 Isso evita que um processador fique sobrecarregado ("Data Skew")

In [ ]:
df_final_preparado = df_final_preparado.repartition("estado")

In [ ]:
print(f"Linhas Olist: {df_final_preparado.count()}")
print(f"Linhas Malha: {df_malha_rodoviaria.count()}")
print(f"Linhas INMET: {df_inmet_preparado.count()}")

Linhas Olist: 99441
Linhas Malha: 646487
Linhas INMET: 26872


Análise Estatística (Diagnóstico de Performance)
Primeiro, vamos entender a distribuição dos atrasos para saber se eles são pontuais ou sistêmicos.

In [ ]:
import os
# Isso vai listar todos os arquivos na pasta atual para você conferir se o seu parquet está lá
print("Arquivos na pasta atual:", os.listdir(base_path))

Arquivos na pasta atual: ['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'southeast.csv', 'stations.csv', 'orders_cleaned.parquet', 'geolocation_cleaned.parquet', 'inmet_southeast_cleaned.parquet', 'inmet_southeast_cleaned', 'stations_cleaned', 'Relatório: Diagnóstico e Predição de Gargalos Logísticos.gdoc', 'tri', 'grafo_rmsp.graphml', 'malha_rodoviaria_particionada', 'dados_dashboard.csv', 'customers_cleaned.parquet', 'items_cleaned.parquet', 'reviews_cleaned.parquet', 'inmet_southeast_final.parquet', 'df_completo_final.parquet', 'df_final_integrado.parquet', 'stations_final.parquet', 'malha_rodoviaria_limpa.parquet']


In [ ]:
# Verificação de segurança
tabelas_esperadas = ["df_final", "df_malha_rodoviaria", "df_inmet"]

for nome in tabelas_esperadas:
    try:
        # Se a variável existe, printa a contagem de linhas
        print(f"✅ {nome} existe! Total de linhas: {globals()[nome].count()}")
    except NameError:
        print(f"❌ {nome} NÃO foi encontrada na memória.")
    except Exception as e:
        print(f"⚠️ {nome} encontrada, mas deu erro: {e}")

✅ df_final existe! Total de linhas: 99441
✅ df_malha_rodoviaria existe! Total de linhas: 646487
⚠️ df_inmet encontrada, mas deu erro: 'df_inmet'


In [ ]:
from pyspark.sql.functions import broadcast, col, to_date, datediff, when, lit

# A base 'df_final' já foi carregada e preparada na célula YtjW8lMUulbS.
# Ela não possui as colunas 'dias_atraso' e 'is_gargalo' ainda.

# 1. Recriar as métricas de SLA (dias_atraso, is_gargalo) no df_final
# Este trecho é similar ao que estava na célula 4qiJ5jNxVoKr
df_final = df_final \
    .withColumn("dt_entrega_real", to_date(when(col("order_delivered_customer_date").isNull(), lit(None)).otherwise(col("order_delivered_customer_date")))) \
    .withColumn("dt_entrega_estimada", to_date(when(col("order_estimated_delivery_date").isNull(), lit(None)).otherwise(col("order_estimated_delivery_date")))) \
    .withColumn("dias_atraso", datediff(col("dt_entrega_real"), col("dt_entrega_estimada")))

df_final = df_final.withColumn("is_gargalo", \
    when(col("dias_atraso") > 0, 1).otherwise(0))

# 2. Seleção de colunas essenciais para o JOIN (Isso evita erros de coluna)
df_olist_light = df_final.select("order_id", "estado", "dias_atraso", "is_gargalo", F.to_date("order_purchase_timestamp").alias("data"))
df_malha_light = df_malha_rodoviaria.select("estado", "speed_kph")
df_inmet_light = df_inmet_preparado.select("estado", "data", "media_chuva_mm", "max_chuva_mm")

# 3. Integração com Broadcast (Otimizada)
df_analise_total = df_olist_light \
    .join(broadcast(df_malha_light), "estado", "left") \
    .join(broadcast(df_inmet_light), ["estado", "data"], "left") \
    .cache()

# 4. Materialização
print(f"🚀 Integração concluída! Total de linhas: {df_analise_total.count()}")

{"ts": "2026-07-08 18:04:18.141", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `estado` cannot be resolved. Did you mean one of the following? [`price`, `order_id`, `dias_atraso`, `is_gargalo`, `seller_id`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor27.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o406.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `estado` cannot be resolved. Did you mean one of the following? [`price`, `order_id`, `dias_atraso`, `is_gargalo`, `seller_id`]. SQLSTATE: 42703;\n'Project [order_id#272, 'estado, dias_atraso#792, is_gargalo#793, to_date(order_purchase_timestamp#275, None, Some(Et

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `estado` cannot be resolved. Did you mean one of the following? [`price`, `order_id`, `dias_atraso`, `is_gargalo`, `seller_id`]. SQLSTATE: 42703;
'Project [order_id#272, 'estado, dias_atraso#792, is_gargalo#793, to_date(order_purchase_timestamp#275, None, Some(Etc/UTC), true) AS data#794]
+- Project [customer_id#273, order_id#272, order_status#274, order_purchase_timestamp#275, order_approved_at#276, order_delivered_carrier_date#277, order_delivered_customer_date#278, order_estimated_delivery_date#279, order_item_id#281L, product_id#282, seller_id#283, shipping_limit_date#284, price#285, freight_value#286, customer_unique_id#288, customer_zip_code_prefix#289, customer_city#290, customer_state#291, dt_entrega_real#790, dt_entrega_estimada#791, dias_atraso#792, CASE WHEN (dias_atraso#792 > 0) THEN 1 ELSE 0 END AS is_gargalo#793]
   +- Project [customer_id#273, order_id#272, order_st

In [ ]:
import pandas as pd
import os
from pyspark.sql import SparkSession

# Desativar configurações de compatibilidade Spark anteriores que não funcionaram
# A nova abordagem será ler com Pandas e converter para PySpark.
# spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")
# spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "LEGACY")
# spark.conf.set("spark.sql.legacy.parquet.int96.canConvertToTimestamp", "true")
# spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

# Carregando sua base tratada que já tem os nulos removidos e datas convertidas
# PRIMEIRA TENTATIVA: Ler com Pandas para contornar o problema de compatibilidade do Spark com timestamps
pd_df_final = pd.read_parquet(os.path.join(base_path, 'df_completo_final.parquet'))

# SEGUNDA TENTATIVA: Converter o DataFrame do Pandas para um DataFrame do PySpark
df_final = spark.createDataFrame(pd_df_final)

# Verificando se os nomes das colunas estão corretos (essencial para o join)
df_final = df_final.withColumnRenamed("customer_state", "estado")
print("✅ Base carregada via Pandas e convertida para PySpark. Colunas disponíveis:", df_final.columns)

In [ ]:
# Definindo o caminho onde você quer salvar o arquivo final
# Usamos a mesma pasta de dados para manter o projeto organizado
caminho_save = "/content/drive/MyDrive/projeto_delivery_logistica/data/df_final_integrado.parquet"

# Salvando o DataFrame como Parquet
# O modo 'overwrite' garante que, se o arquivo já existir, ele será substituído pelo novo
df_final.write.mode("overwrite").parquet(caminho_save)

print(f"Arquivo salvo com sucesso em: {caminho_save}")

In [ ]:
# Preenche valores nulos em review_score, se necessário
df_final = df_final.fillna({'review_score': -1})

In [ ]:
#Agora você pode visualizar as colunas relevantes
from pyspark.sql.functions import col

df_final.select(
    "order_id",
    "tempo_entrega_dias",
    "atraso_entrega_dias",
    "media_chuva_mm",
    "review_score",
    "geolocation_lat",
    "geolocation_lng",
    "customer_state"
).show(10)

In [ ]:
# 4. Verifica se o df_final (enriquecido) está disponível
if 'df_final' not in locals() or df_final is None:
    print("❌ Erro: 'df_final' (DataFrame PySpark enriquecido) não encontrado na sessão.")
    print("Por favor, execute as células de 's-jpxiWGIsAR' até '_H4or922JjWm' para enriquecer o 'df_final' antes de executar esta célula.")
    raise NameError("'df_final' (enriquecido) não definido.")

   ## 8. de Variáveis e Correlação
Análise estatística para validar se o aumento da pluviosidade (chuva) impacta o 'atraso_entrega_dias'.

In [ ]:
# Importar bibliotecas
import seaborn as sns
import matplotlib.pyplot as plt

# Selecionar colunas numéricas, incluindo as novas métricas de interesse
# Certifique-se de que esses nomes correspondam exatamente aos do seu df_final_completo
colunas_analise = [
    'media_chuva_mm', 'atraso_entrega_dias', 'tempo_entrega_dias',
    'review_score', 'geolocation_lat', 'geolocation_lng'
]

In [ ]:
# 1. Cria uma amostra representativa (10% dos dados)
df_amostra = df_final_completo.sample(withReplacement=False, fraction=0.1, seed=42)

# 2. Agrupa as colunas da amostra no vetor
assembler = VectorAssembler(inputCols=colunas_analise, outputCol="features_vector")
df_vetor = assembler.transform(df_amostra)

# 3. Calcula a matriz de correlação usando a amostra
matriz_spark = Correlation.corr(df_vetor, "features_vector").head()[0]

In [ ]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import SparkSession # Add this import
import sys # Para sys.exit() ou similar para parar a execução

# 1. Agrupa as colunas em um único vetor (necessário para o Spark)
assembler = VectorAssembler(inputCols=colunas_analise, outputCol="features_vector", handleInvalid="skip")
df_vetor = assembler.transform(df_final_completo)

In [ ]:
# 2. Calcula a matriz de correlação usando o motor do Spark
matriz_spark = Correlation.corr(df_vetor, "features_vector").head()[0]

In [ ]:
# 3. Exibe o resultado (matriz de correlação completa)
print(matriz_spark)

In [ ]:
# 1. Converte a matriz do Spark para DataFrame do Pandas
import pandas as pd
matriz_dados = matriz_spark.toArray()
df_matriz_pandas = pd.DataFrame(matriz_dados, columns=colunas_analise, index=colunas_analise)

# 2. Criar o mapa de calor
plt.figure(figsize=(10, 8))
sns.heatmap(df_matriz_pandas, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Mapa de Calor: Correlação entre Clima, Logística e Localização")
plt.show()

#🚀 9. Análise Preditiva de Atraso Logístico (Machine Learning)
Nesta etapa final do projeto, utilizamos Machine Learning para transformar dados históricos em inteligência preditiva. Nosso objetivo é treinar um modelo capaz de antecipar a probabilidade de um pedido sofrer atraso na entrega.

Preparação: Definindo o Target e as Features
Precisamos transformar nossa variável de atraso em um rótulo binário (0 ou 1).

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import when

# 1. Criar coluna 'target': 1 se atrasou (atraso_entrega_dias > 0), 0 caso contrário
df_ml = df_final.withColumn("target", when(col("atraso_entrega_dias") > 0, 1).otherwise(0))

# 2. Selecionar as colunas (features) que alimentam o modelo
# Usaremos métricas climáticas e temporais
feature_cols = ['media_chuva_mm', 'tempo_entrega_dias', 'geolocation_lat', 'geolocation_lng']

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_model = assembler.transform(df_ml)

# Dividir em Treino (80%) e Teste (20%) para validação profissional
train_data, test_data = df_model.randomSplit([0.8, 0.2], seed=42)

In [ ]:
import pandas as pd
import os

# Defina o caminho base (o mesmo que você usou anteriormente)
BASE_PATH = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

# Carrega o arquivo final preparado para o dashboard
df_dashboard = pd.read_csv(os.path.join(BASE_PATH, 'dados_dashboard.csv'))

# Exibe as colunas para verificar se review_score, lat e lng estão lá
print("Colunas disponíveis no arquivo de dashboard:")
print(df_dashboard.columns.tolist())

# Opcional: exibe uma amostra dos dados para confirmar que não estão vazios
print("\nAmostra dos dados:")
display(df_dashboard.head())

Treinamento do Modelo (Random Forest)
O Random Forest é robusto, difícil de causar overfitting e traz ótimos resultados iniciais.

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

# Configurar o modelo
rf = RandomForestClassifier(labelCol="target", featuresCol="features", numTrees=100)

# Treinar o modelo
model = rf.fit(train_data)

print("Modelo treinado com sucesso!")

Avaliação de Performance
Não basta treinar, precisamos saber se o modelo acerta. Usaremos a Matriz de Confusão (implicitamente via BinaryClassificationEvaluator).

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Fazer previsões no conjunto de teste
predictions = model.transform(test_data)

# Avaliar a precisão (Area Under ROC)
evaluator = BinaryClassificationEvaluator(labelCol="target", rawPredictionCol="rawPrediction")
auc = evaluator.evaluate(predictions)

print(f"Desempenho do Modelo (AUC): {auc:.2f}")

# Mostrar exemplos onde o modelo previu o atraso
predictions.select("target", "prediction", "probability").show(10)

In [ ]:
# Extrair as importâncias do modelo Random Forest
importances = model.featureImportances

# Mapear as importâncias com os nomes das colunas
feature_names = ['media_chuva_mm', 'tempo_entrega_dias', 'geolocation_lat', 'geolocation_lng']
feature_importance_list = list(zip(feature_names, importances))

# Exibir os resultados de forma ordenada
sorted_importances = sorted(feature_importance_list, key=lambda x: x[1], reverse=True)

print("Importância das Variáveis:")
for name, importance in sorted_importances:
    print(f"{name}: {importance:.4f}")

In [ ]:
import pandas as pd
import os

# Define o caminho base onde seus arquivos estão armazenados
BASE_PATH = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

# Carrega os DataFrames necessários como pandas DataFrames
# 'df_pedidos' provavelmente se refere ao 'df_orders' tratado
df_pedidos = pd.read_parquet(os.path.join(BASE_PATH, 'orders_cleaned.parquet'))
# 'df_clientes' provavelmente se refere ao 'df_customers' tratado
df_clientes = pd.read_parquet(os.path.join(BASE_PATH, 'customers_cleaned.parquet'))
# Carrega o DataFrame de reviews. Assumindo que o arquivo CSV original está na pasta.
try:
    df_reviews = pd.read_csv(os.path.join(BASE_PATH, 'olist_order_reviews_dataset.csv'))
except FileNotFoundError:
    print("Erro: 'olist_order_reviews_dataset.csv' não encontrado. Por favor, verifique o caminho ou nome do arquivo.")
    # Cria um DataFrame vazio para evitar que o código falhe completamente
    df_reviews = pd.DataFrame(columns=['order_id', 'review_score'])

# 1. Trazer a nota de satisfação (review_score)
# Certifique-se de que a coluna 'order_id' existe em ambos
df_final = df_pedidos.merge(df_reviews[['order_id', 'review_score']], on='order_id', how='left')

# 2. Trazer a localização (cliente_estado ou cliente_regiao)
# Se você tiver uma coluna 'customer_id' na tabela de pedidos e na tabela de clientes
df_final = df_final.merge(df_clientes[['customer_id', 'customer_state']], on='customer_id', how='left')

# 3. Preencher valores vazios, se houver
df_final['review_score'] = df_final['review_score'].fillna(0) # Nota 0 se não houver avaliação
df_final['customer_state'] = df_final['customer_state'].fillna('Não informado')

# 4. Salvar novamente para atualizar o seu arquivo no Drive
df_final.to_csv('dash - dados_dashboard.csv', index=False)


In [ ]:
# Lista de dataframes que você está manipulando
dfs = {
    "df_pedidos": df_pedidos, # Substitua pelo nome correto da sua variável
    "df_reviews": df_reviews, # Se houver uma tabela de avaliações
    "df_customers": df_customers # Se houver uma tabela de clientes com local
}

for nome, df in dfs.items():
    try:
        print(f"Colunas em {nome}: {df.columns}")
    except:
        print(f"{nome} não está carregado.")

In [ ]:
# Carregue o seu dataframe (ajuste o caminho se necessário)
df_final = spark.read.parquet("/content/drive/MyDrive/projeto_delivery_logistica/data/df_final_integrado.parquet")

# Converte para Pandas e salva como CSV
df_pandas = df_final.toPandas()
df_pandas.to_csv("/content/drive/MyDrive/projeto_delivery_logistica/data/dados_dashboard.csv", index=False)

print("Arquivo salvo como CSV no seu Drive!")

In [ ]:
import pandas as pd
import os

# Defina o caminho base (o mesmo que você usou anteriormente)
BASE_PATH = '/content/drive/MyDrive/projeto_delivery_logistica/data/'

# Carrega o arquivo final preparado para o dashboard
df_dashboard = pd.read_csv(os.path.join(BASE_PATH, 'dados_dashboard.csv'))

# Exibe as colunas para verificar se review_score, lat e lng estão lá
print("Colunas disponíveis no arquivo de dashboard:")
print(df_dashboard.columns.tolist())

# Opcional: exibe uma amostra dos dados para confirmar que não estão vazios
print("\nAmostra dos dados:")
display(df_dashboard.head())